# MATH 161A Report 3 — Notebook 2
## 1. Introduction and research question
**Adversarial examples, FGSM training, and PGD robustness on MNIST**

*To what extent does FGSM adversarial training improve a CNN's robustness to adversarial perturbations, and does that robustness generalize beyond the perturbations encountered during training?*

This is a reduced-scale, reproduction-inspired investigation of Goodfellow, Shlens, and Szegedy (2015), [Explaining and Harnessing Adversarial Examples](https://arxiv.org/abs/1412.6572), and Madry et al. (2018), [Towards Deep Learning Models Resistant to Adversarial Attacks](https://arxiv.org/abs/1706.06083). It is not an exact replication.

Run cells in order in the existing VSCode-connected Kaggle kernel. No results have been generated. This notebook neither installs packages nor writes checkpoints or result files. Provision torchvision-compatible MNIST files in advance and set `DATA_ROOT` below; loading uses `download=False`. The approximate 60-minute GPU budget is a planning target, not a runtime guarantee.

## 2. Mathematical background placeholders
**Student writing placeholder:** Define the classification objective, input-gradient linearization underlying FGSM, the mixed training objective, and the projected inner maximization used by PGD. Supply derivations and paper-specific context independently.

Implementation specification (not a derivation): inputs remain in $[0,1]$; attacks are untargeted and maximize true-label cross-entropy. FGSM uses $x_{adv}=\mathrm{clip}_{[0,1]}(x+\epsilon\,\mathrm{sign}(\nabla_x\mathcal L))$. PGD takes 10 sign-gradient steps, projecting into $B_\infty(x,\epsilon)\cap[0,1]^d$ after every step. Its step size is $\alpha=\epsilon/4$ for positive epsilon. Zero epsilon returns the original image exactly.

## 3. Dataset and experimental setup
Use only `ToTensor()` preprocessing. Split the original 60,000 training images into 55,000 training and 5,000 clean validation images. The official 10,000-image test set is accessed for evaluation only after training. One seed controls initialization, partitioning, ordering, and attack randomness; separate local generators prevent unrelated random draws from changing these decisions. Deterministic settings improve repeatability on the same software/hardware; cross-platform bitwise equivalence is not promised.

In [ ]:
# Import packages and configure the single-seed experiment.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import copy
import random
import time
from pathlib import Path
from contextlib import contextmanager
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms
from IPython.display import display

SEED = 42
EPOCHS = 12
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
LEARNING_RATE = 0.001
EPS_TRAIN = 0.20
EPSILONS = (0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30)
PGD_STEPS = 10
TOL = 1e-6
DATA_ROOT = Path("/kaggle/input/mnist-torchvision")  # Parent of MNIST/raw/.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything():
  random.seed(SEED)
  np.random.seed(SEED)
  torch.manual_seed(SEED)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
  torch.backends.cudnn.benchmark = False
  torch.backends.cudnn.deterministic = True
  torch.use_deterministic_algorithms(True)
  if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

seed_everything()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print(f"Seed={SEED}; device={device}; torch={torch.__version__}; "
      f"torchvision={torchvision.__version__}; NumPy={np.__version__}")
if device.type == "cuda":
  print(f"GPU: {torch.cuda.get_device_name(0)}; CUDA: {torch.version.cuda}")
else:
  print("CPU selected; the GPU runtime target does not apply.")

In [ ]:
# Load only the original training split and verify the fixed partitions.
train_source = datasets.MNIST(
  root=str(DATA_ROOT), train=True, transform=transforms.ToTensor(), download=False
)
assert len(train_source) == 60000
split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_source), generator=split_generator)
val_indices = permutation[:5000].tolist()
train_indices = permutation[5000:].tolist()
assert len(set(train_indices) & set(val_indices)) == 0
assert set(train_indices) | set(val_indices) == set(range(60000))
train_data = Subset(train_source, train_indices)
val_data = Subset(train_source, val_indices)

def make_loader(data, training=False):
  return DataLoader(
    data, batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE,
    shuffle=training, drop_last=False, num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=torch.Generator().manual_seed(SEED),
  )

val_loader = make_loader(val_data)
print(f"Training={len(train_data)}; validation={len(val_data)}; test reserved=10000")

## 4. CNN architecture and training configuration
Two 3×3 padded convolutional layers (32 then 64 channels), each followed by ReLU and 2×2 max pooling, feed a 128-unit ReLU layer and ten logits. No dropout, batch normalization, or input normalization is used. Both models receive identical initial weights, Adam (learning rate 0.001, default betas, zero weight decay), batches of 128, and 12 epochs with constant learning rate. No tuning or checkpoint selection is performed; final epoch weights are evaluated.

Attack functions temporarily enter evaluation mode, restore the previous mode, and use `autograd.grad` only for inputs. Returned images are detached. Parameter gradients are neither populated nor cleared by attack generation. The architecture's training/evaluation forwards are identical.

In [ ]:
# Define the compact CNN and preserve a shared initialization.
class CompactCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
      nn.Linear(128, 10),
    )

  def forward(self, x):
    return self.layers(x)

seed_everything()
initial_model = CompactCNN()
initial_state = copy.deepcopy(initial_model.state_dict())
models = {name: CompactCNN().to(device) for name in ("Standard", "FGSM trained")}
for model in models.values():
  model.load_state_dict(initial_state)
for key in initial_state:
  assert torch.equal(models["Standard"].state_dict()[key],
                      models["FGSM trained"].state_dict()[key])
print(f"Trainable parameters per CNN: {sum(p.numel() for p in initial_model.parameters()):,}")
del initial_model

In [ ]:
# Implement detached FGSM and random-start projected gradient attacks.
@contextmanager
def attack_mode(model):
  was_training = model.training
  model.eval()
  try:
    with torch.enable_grad():
      yield
  finally:
    model.train(was_training)

def fgsm(model, x, y, epsilon):
  if epsilon < 0:
    raise ValueError("epsilon must be nonnegative")
  if epsilon == 0:
    return x.detach().clone()
  with attack_mode(model):
    z = x.detach().clone().requires_grad_(True)
    gradient, = torch.autograd.grad(F.cross_entropy(model(z), y), z)
    return (z + epsilon * gradient.sign()).clamp(0, 1).detach()

def pgd(model, x, y, epsilon, generator=None):
  if epsilon < 0:
    raise ValueError("epsilon must be nonnegative")
  if epsilon == 0:
    return x.detach().clone()
  alpha = epsilon / 4
  original = x.detach()
  lower = (original - epsilon).clamp(0, 1)
  upper = (original + epsilon).clamp(0, 1)
  noise = torch.empty_like(original).uniform_(
    -epsilon, epsilon, generator=generator
  )
  z = (original + noise).clamp(0, 1)
  # Retain per-image highest-loss candidates, including clean and random start.
  with attack_mode(model):
    with torch.no_grad():
      best = original.clone()
      best_loss = F.cross_entropy(model(best), y, reduction="none")
    for step in range(PGD_STEPS + 1):
      z = z.detach().requires_grad_(True)
      losses = F.cross_entropy(model(z), y, reduction="none")
      with torch.no_grad():
        improved = losses > best_loss
        best[improved] = z.detach()[improved]
        best_loss = torch.maximum(best_loss, losses.detach())
      if step < PGD_STEPS:
        gradient, = torch.autograd.grad(losses.sum(), z)
        z = torch.maximum(lower, torch.minimum(upper, z + alpha * gradient.sign()))
  return best.detach()

def check_images(original, adversarial, epsilon):
  assert original.ndim == 4 and original.shape[1:] == (1, 28, 28)
  assert adversarial.shape == original.shape
  assert torch.isfinite(adversarial).all()
  assert original.min().item() >= -TOL and original.max().item() <= 1 + TOL
  assert adversarial.min().item() >= -TOL and adversarial.max().item() <= 1 + TOL
  assert (adversarial - original).abs().max().item() <= epsilon + TOL
  if epsilon == 0:
    assert torch.equal(original, adversarial)

## 5. Baseline and adversarial training
Each model uses its own freshly seeded training loader, yielding identical epoch-wise sample order. In mixed training, half the batch positions are selected uniformly using an independent generator and replaced with on-the-fly FGSM images at epsilon 0.20; the remaining images stay clean. Every source sample contributes once per epoch. This holds the number of updates and source examples constant, although attack computation adds cost.

Training metrics describe the inputs used for each update (clean for Standard; approximately 50/50 mixed for FGSM trained), measured before that update. Validation metrics always use clean images. The curves and summary report sample-weighted metrics and measured training time.

In [ ]:
# Define clean evaluation and controlled training with mixed-input metrics.
@torch.no_grad()
def evaluate_clean(model, loader):
  model.eval()
  total_loss, correct, count = 0.0, 0, 0
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    logits = model(x)
    assert logits.shape == (len(y), 10)
    total_loss += F.cross_entropy(logits, y, reduction="sum").item()
    correct += (logits.argmax(1) == y).sum().item()
    count += len(y)
  return {"loss": total_loss / count, "accuracy": correct / count, "n": count}

def train_model(model, adversarial=False):
  seed_everything()
  loader = make_loader(train_data, training=True)
  mixture_generator = torch.Generator().manual_seed(SEED)
  optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
  history = []
  if device.type == "cuda":
    torch.cuda.synchronize()
  started = time.perf_counter()
  for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, correct, count, adversarial_count = 0.0, 0, 0, 0
    for x, y in loader:
      x, y = x.to(device), y.to(device)
      optimizer.zero_grad(set_to_none=True)
      if adversarial:
        positions = torch.randperm(len(y), generator=mixture_generator)[:len(y) // 2]
        positions = positions.to(device)
        mixed = x.clone()
        mixed[positions] = fgsm(model, x[positions], y[positions], EPS_TRAIN)
        x = mixed.detach()
        adversarial_count += len(positions)
      logits = model(x)
      loss = F.cross_entropy(logits, y)
      loss.backward()
      optimizer.step()
      loss_sum += loss.item() * len(y)
      correct += (logits.detach().argmax(1) == y).sum().item()
      count += len(y)
    validation = evaluate_clean(model, val_loader)
    record = {
      "epoch": epoch, "train_loss": loss_sum / count,
      "train_accuracy": correct / count, "val_loss": validation["loss"],
      "val_accuracy": validation["accuracy"],
      "adversarial_fraction": adversarial_count / count,
    }
    history.append(record)
    kind = "mixed" if adversarial else "clean"
    print(f"{kind:5s} {epoch:02d}/{EPOCHS}: train loss={record['train_loss']:.4f}, "
          f"acc={100 * record['train_accuracy']:.2f}%; "
          f"clean val loss={record['val_loss']:.4f}, "
          f"acc={100 * record['val_accuracy']:.2f}%")
  optimizer.zero_grad(set_to_none=True)
  if device.type == "cuda":
    torch.cuda.synchronize()
  return pd.DataFrame(history), time.perf_counter() - started

histories, training_seconds = {}, {}
for name, model in models.items():
  histories[name], training_seconds[name] = train_model(
    model, adversarial=(name == "FGSM trained")
  )

In [ ]:
# Plot labeled training and clean-validation curves and summarize final epochs.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for name, history in histories.items():
  for row, metric in enumerate(("loss", "accuracy")):
    scale = 100 if metric == "accuracy" else 1
    kind = "mixed" if name == "FGSM trained" else "clean"
    axes[row, 0].plot(history.epoch, scale * history[f"train_{metric}"],
                      label=f"{name} ({kind} inputs)")
    axes[row, 1].plot(history.epoch, scale * history[f"val_{metric}"], label=name)
for ax, title, ylabel in zip(axes.flat,
    ("Training loss", "Clean validation loss", "Training accuracy", "Clean validation accuracy"),
    ("Cross-entropy", "Cross-entropy", "Accuracy (%)", "Accuracy (%)")):
  ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
  ax.set_xticks(range(1, EPOCHS + 1))
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
training_summary = pd.DataFrame([
  {"Model": name, "Training inputs": "50/50 mixed" if name == "FGSM trained" else "Clean",
    "Final train loss": h.iloc[-1].train_loss,
    "Final train accuracy (%)": 100 * h.iloc[-1].train_accuracy,
    "Final clean val loss": h.iloc[-1].val_loss,
    "Final clean val accuracy (%)": 100 * h.iloc[-1].val_accuracy,
    "Training minutes": training_seconds[name] / 60}
  for name, h in histories.items()
])
display(training_summary.round(3))

## 6. FGSM robustness experiment
Only now load the official test split. Full-set FGSM evaluation uses all 10,000 images, including originally misclassified images in the denominator. Each model is attacked using its own gradients. A fixed random 1,000-image subset is reserved for matched FGSM/PGD comparisons; its original test indices are retained in memory. All loaders preserve order and keep incomplete final batches.

The following checks execute only when this notebook is run: shapes, pixel bounds, perturbation norms, zero-epsilon identities, unchanged parameter gradients, restored model mode, and partition/selection integrity.

In [ ]:
# Construct final-evaluation loaders and verify attack behavior without model updates.
test_source = datasets.MNIST(
  root=str(DATA_ROOT), train=False, transform=transforms.ToTensor(), download=False
)
assert train_source.train is True and test_source.train is False
assert train_data.dataset is train_source and val_data.dataset is train_source
assert len(test_source) == 10000
subset_indices = torch.randperm(10000, generator=torch.Generator().manual_seed(SEED))[:1000].tolist()
assert len(set(subset_indices)) == 1000
assert all(0 <= i < 10000 for i in subset_indices)
test_subset = Subset(test_source, subset_indices)
full_test_loader = make_loader(test_source)
subset_loader = make_loader(test_subset)
assert subset_loader.dataset is test_subset
assert subset_loader.dataset.indices == subset_indices

probe_x, probe_y = next(iter(val_loader))
probe_x, probe_y = probe_x[:8].to(device), probe_y[:8].to(device)
for model in models.values():
  model.eval()
  saved_parameters = [p.detach().clone() for p in model.parameters()]
  # Nonzero sentinels verify attacks preserve existing gradients as well as values.
  for p in model.parameters():
    assert p.grad is None
    p.grad = torch.ones_like(p)
  for epsilon in (0.0, EPS_TRAIN):
    for attack in (fgsm, pgd):
      attacked = attack(model, probe_x, probe_y, epsilon)
      check_images(probe_x, attacked, epsilon)
      assert not attacked.requires_grad and not model.training
      assert model(attacked).shape == (len(probe_y), 10)
      assert all(torch.equal(p.grad, torch.ones_like(p)) for p in model.parameters())
  assert all(torch.equal(p, old) for p, old in zip(model.parameters(), saved_parameters))
  for p in model.parameters():
    p.grad = None
  model.train()
  fgsm(model, probe_x, probe_y, EPS_TRAIN)
  assert model.training
  assert all(p.grad is None for p in model.parameters())
  model.eval()
print("Partition, shape, constraint, zero-attack, and gradient checks passed.")

In [ ]:
# Evaluate white-box attacks while retaining ordered per-image predictions.
def evaluate_attack(model, loader, attack, epsilon):
  model.eval()
  predictions, labels = [], []
  # Reset for every call: matched random starts across models at each epsilon.
  generator = torch.Generator(device=device).manual_seed(SEED)
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    if attack == "FGSM":
      attacked = fgsm(model, x, y, epsilon)
    elif attack == "PGD":
      attacked = pgd(model, x, y, epsilon, generator=generator)
    else:
      raise ValueError("Unknown attack")
    check_images(x, attacked, epsilon)
    with torch.no_grad():
      logits = model(attacked)
      assert logits.shape == (len(y), 10)
      predictions.append(logits.argmax(1).cpu())
      labels.append(y.cpu())
  predictions, labels = torch.cat(predictions), torch.cat(labels)
  assert len(labels) == len(loader.dataset)
  return {"predictions": predictions, "labels": labels,
          "accuracy": (predictions == labels).float().mean().item(), "n": len(labels)}

full_fgsm, clean_full = {}, {}
for name, model in models.items():
  clean_full[name] = evaluate_clean(model, full_test_loader)
  for epsilon in EPSILONS:
    result = evaluate_attack(model, full_test_loader, "FGSM", epsilon)
    assert torch.equal(result["labels"], test_source.targets)
    if epsilon == 0:
      assert abs(result["accuracy"] - clean_full[name]["accuracy"]) < TOL
    full_fgsm[name, epsilon] = result
fgsm_table = pd.DataFrame({
  name: [100 * full_fgsm[name, e]["accuracy"] for e in EPSILONS]
  for name in models
}, index=pd.Index(EPSILONS, name="Epsilon"))
display(fgsm_table.round(2).rename_axis(columns="Full-test FGSM accuracy (%)"))
ax = fgsm_table.plot(marker="o", figsize=(8, 5))
ax.set(title="White-box FGSM — all 10,000 test images", xlabel="Epsilon (pixel scale)",
        ylabel="Accuracy (%)", ylim=(0, 100))
plt.tight_layout()
plt.show()

In [ ]:
# Display predetermined test examples with signed perturbations on a fixed scale.
example_indices = subset_indices[:4]  # Chosen before inspecting predictions.
example_x = torch.stack([test_source[i][0] for i in example_indices]).to(device)
example_y = torch.tensor([test_source[i][1] for i in example_indices], device=device)
fig, axes = plt.subplots(2 * len(example_indices), 3, figsize=(10, 18))
for model_number, (name, model) in enumerate(models.items()):
  adversarial = fgsm(model, example_x, example_y, EPS_TRAIN)
  check_images(example_x, adversarial, EPS_TRAIN)
  with torch.no_grad():
    clean_predictions = model(example_x).argmax(1).cpu().tolist()
    adv_predictions = model(adversarial).argmax(1).cpu().tolist()
  for j, test_index in enumerate(example_indices):
    row = model_number * len(example_indices) + j
    delta = (adversarial[j] - example_x[j]).detach().cpu().squeeze()
    axes[row, 0].imshow(example_x[j].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{name}: test #{test_index}\nTrue {example_y[j].item()}, clean pred {clean_predictions[j]}", fontsize=9)
    perturbation = axes[row, 1].imshow(delta, cmap="seismic", vmin=-EPS_TRAIN, vmax=EPS_TRAIN)
    axes[row, 1].set_title(f"Signed delta; max |delta|={delta.abs().max():.3f}", fontsize=9)
    axes[row, 2].imshow(adversarial[j].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title(f"FGSM epsilon={EPS_TRAIN:.2f}; pred {adv_predictions[j]}", fontsize=9)
    for ax in axes[row]:
      ax.set_xticks([])
      ax.set_yticks([])
fig.suptitle("Fixed examples (not selected for attack success); delta color exaggerates visibility")
fig.tight_layout(rect=(0, 0, 0.9, 0.97))
cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
fig.colorbar(perturbation, cax=cbar_ax, label="Actual signed pixel change (fixed ±0.20 range)")
plt.show()

## 7. PGD robustness experiment
Evaluate 10-step, one-random-start PGD on the fixed 1,000-image subset for every epsilon and model. Step size is epsilon/4, permitting traversal of the constraint region within ten steps. The random start samples uniformly in the centered perturbation cube before pixel clipping. PGD retains the per-image highest-cross-entropy candidate among the clean image, random start, and all ten iterates; this avoids discarding an earlier higher-loss iterate. Highest cross-entropy is not necessarily identical to classification failure.

Recompute FGSM on this exact ordered subset. Curves report unconditional accuracy, not attack success conditional on initially correct predictions. One restart and ten steps are a finite attack budget; these are empirical attack accuracies, not certified robustness estimates.

In [ ]:
# Evaluate matched-subset FGSM and PGD and verify all zero-epsilon predictions.
matched, clean_subset = {}, {}
expected_labels = test_source.targets[torch.tensor(subset_indices)]
for name, model in models.items():
  clean_subset[name] = evaluate_clean(model, subset_loader)
  for epsilon in EPSILONS:
    for attack in ("FGSM", "PGD"):
      result = evaluate_attack(model, subset_loader, attack, epsilon)
      assert result["n"] == 1000
      assert torch.equal(result["labels"], expected_labels)
      if epsilon == 0:
        assert abs(result["accuracy"] - clean_subset[name]["accuracy"]) < TOL
        assert torch.equal(result["predictions"],
                            full_fgsm[name, 0.0]["predictions"][subset_indices])
      matched[name, attack, epsilon] = result
  for epsilon in EPSILONS:
    assert torch.equal(matched[name, "FGSM", epsilon]["predictions"],
                        full_fgsm[name, epsilon]["predictions"][subset_indices])
pgd_table = pd.DataFrame({
  name: [100 * matched[name, "PGD", e]["accuracy"] for e in EPSILONS]
  for name in models
}, index=pd.Index(EPSILONS, name="Epsilon"))
display(pgd_table.round(2).rename_axis(columns="Matched-subset PGD accuracy (%)"))
ax = pgd_table.plot(marker="o", figsize=(8, 5))
ax.set(title="White-box PGD-10, one random start — fixed 1,000 test images",
        xlabel="Epsilon (pixel scale)", ylabel="Accuracy (%)", ylim=(0, 100))
plt.tight_layout()
plt.show()

## 8. Comparative results
All direct attack comparisons below use the same 1,000 test images. Tables include matched FGSM and PGD accuracies and their difference in percentage points (FGSM minus PGD) at every epsilon. The grouped chart compares clean, FGSM, and PGD accuracy at epsilon 0.20, with clean serving as the zero-perturbation reference.

**Student analysis placeholder:** Quantify the clean/robust tradeoff, training-method differences, and matched FGSM–PGD gaps after execution. Do not assume PGD accuracy must be lower at every epsilon or impose monotonic curves; inspect unexpected behavior without tuning on the test set.

In [ ]:
# Tabulate matched attack accuracies and the FGSM–PGD gap at every epsilon.
comparison = pd.DataFrame([
  {"Model": name, "Epsilon": e, "N": 1000,
    "FGSM accuracy (%)": 100 * matched[name, "FGSM", e]["accuracy"],
    "PGD accuracy (%)": 100 * matched[name, "PGD", e]["accuracy"],
    "FGSM minus PGD (pp)": 100 * (matched[name, "FGSM", e]["accuracy"]
                                    - matched[name, "PGD", e]["accuracy"])}
  for name in models for e in EPSILONS
])
display(comparison.round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, name in zip(axes, models):
  rows = comparison[comparison.Model == name]
  ax.plot(rows.Epsilon, rows["FGSM accuracy (%)"], marker="o", label="FGSM")
  ax.plot(rows.Epsilon, rows["PGD accuracy (%)"], marker="s", label="PGD-10")
  ax.set(title=f"{name}: matched 1,000 test images", xlabel="Epsilon (pixel scale)",
          ylabel="Accuracy (%)", ylim=(0, 100))
  ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Compare clean and attacked accuracy at the prespecified training epsilon.
summary_020 = pd.DataFrame([
  {"Model": name, "N": 1000,
    "Clean (%)": 100 * clean_subset[name]["accuracy"],
    "FGSM (%)": 100 * matched[name, "FGSM", EPS_TRAIN]["accuracy"],
    "PGD-10 (%)": 100 * matched[name, "PGD", EPS_TRAIN]["accuracy"]}
  for name in models
]).set_index("Model")
display(summary_020.round(2))
ax = summary_020[["Clean (%)", "FGSM (%)", "PGD-10 (%)"]].plot.bar(
  figsize=(8, 5), rot=0, width=0.75
)
ax.set(title="Clean versus attacks at epsilon 0.20 — matched 1,000 test images",
        xlabel="Training method", ylabel="Accuracy (%)", ylim=(0, 100))
ax.legend(title="Evaluation condition")
plt.tight_layout()
plt.show()
print(f"Measured total training time: {sum(training_seconds.values()) / 60:.2f} minutes")

## 9. Discussion, limitations, and conclusion placeholders
**Discussion — student writing placeholder:** Answer the research question using the computed tables and plots. Distinguish full-test FGSM evidence from matched-subset attack comparisons. Discuss whether resistance at the training epsilon transfers to other perturbation strengths and to the specified PGD attack.

**Limitations — student writing placeholder:** Address the single initialization/training run; compact architecture and MNIST scope; 1,000-image random subset sampling; final-epoch evaluation; mixed-training metric comparability; and finite attack budget. FGSM training may exhibit gradient masking or catastrophic overfitting; clean validation alone cannot diagnose them. FGSM resistance is insufficient evidence of robustness to stronger attacks. Ten-step, single-start PGD does not establish worst-case robustness, and this notebook does not perform a comprehensive attack audit. Do not infer significance or seed-to-seed variability from this single run.

**Conclusion — student writing placeholder:** Write an evidence-based conclusion independently after executing and reviewing the notebook. No experimental outcomes or academic conclusions are supplied here.

### References
- Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015). *Explaining and Harnessing Adversarial Examples*. ICLR. https://arxiv.org/abs/1412.6572
- Madry, A., Makelov, A., Schmidt, L., Tsipras, D., & Vladu, A. (2018). *Towards Deep Learning Models Resistant to Adversarial Attacks*. ICLR. https://arxiv.org/abs/1706.06083